# SegOCR — Test/Eval Notebook (paper metrics)

Runs every metric reported in the research paper on a fresh held-out synthetic test set. Designed to be run **after** training is complete — attach either a single-worker checkpoint or the averaged ensemble `ensemble.pth`.

**Setup:**
1. Settings → Accelerator: **GPU T4** (T4 x2 also fine; only one used).
2. Add Data: upload your trained `.pth` checkpoint as a Kaggle Dataset (e.g. `segocr-ensemble-checkpoint`) and attach it. Either `ensemble.pth` (preferred for headline numbers) or any single `averaged_best.pth` works — both load identically.
3. Click **Save Version → Save & Run All**. Takes ~30-40 min (generation ~15 min + eval ~10 min + decode ~5 min).

**Outputs:**
- `/kaggle/working/results.json` — all metrics, machine-readable
- `/kaggle/working/results_paper_table.md` — markdown table for the paper
- `/kaggle/working/per_class_iou.png` — per-class IoU bar chart
- `/kaggle/working/qualitative_gallery.png` — success + failure cases

## 1 / Setup — clone repo + install deps

In [ ]:
import os
if not os.path.isdir('/kaggle/working/segocr'):
    !git clone https://github.com/mauryantitans/SegOCR.git /kaggle/working/segocr
%cd /kaggle/working/segocr
!git pull --quiet
!pip install -q -e .
!pip install -q -r requirements/base.txt
!pip install -q segmentation-models-pytorch

## 2 / Verify GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Settings → Accelerator → GPU T4.')
DEVICE = torch.device('cuda:0')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Mem: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3 / Locate the trained checkpoint

Searches `/kaggle/input/` for `.pth` files. If multiple are found (e.g. you attached all 5 worker checkpoints), they're averaged on the fly into an ensemble — equivalent to running `scripts/average_runs.py` locally.

In [ ]:
import glob

candidates = []
for pat in (
    '/kaggle/input/*/ensemble.pth',
    '/kaggle/input/*/averaged_best.pth',
    '/kaggle/input/*/*.pth',
    '/kaggle/input/*/*/*.pth',
):
    candidates.extend(glob.glob(pat))
candidates = sorted(set(candidates))
if not candidates:
    raise FileNotFoundError(
        'No .pth checkpoints found under /kaggle/input/. Attach a Dataset '
        'containing ensemble.pth (or averaged_best.pth files).'
    )
print(f'Found {len(candidates)} checkpoint(s):')
for p in candidates:
    sz = os.path.getsize(p) / 1e6
    print(f'  {p}  ({sz:.1f} MB)')

CHECKPOINT_PATHS = candidates

## 4 / Generate a held-out test set

1,000 fresh synthetic samples at indices `[80000, 81000)` — outside the range any training worker used (workers 0-4 consumed indices `[0, 80000)`). Generation takes ~10-15 min on Kaggle CPU. Skips automatically if a previous run's output is attached.

In [ ]:
import os, glob, time

TEST_SET_SIZE = 1_000
TEST_INDEX_OFFSET = 80_000
TEST_DIR = '/kaggle/working/test_set'

# Reuse pre-generated test set if attached as a dataset
prev_test_dirs = [
    p for p in glob.glob('/kaggle/input/*/test_set')
    if os.path.isdir(os.path.join(p, 'images'))
]
if prev_test_dirs:
    src = prev_test_dirs[0]
    print(f'Reusing pre-generated test set: {src}')
    TEST_DIR = src
elif (os.path.isdir(f'{TEST_DIR}/images')
      and len(os.listdir(f'{TEST_DIR}/images')) >= TEST_SET_SIZE):
    print(f'Test set already present at {TEST_DIR}; skipping generation.')
else:
    import yaml
    from pathlib import Path
    cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())
    cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
    cfg['generator']['fonts']['cache_path'] = '/kaggle/working/font_cache.json'
    cfg['generator']['fonts']['min_size'] = 40
    cfg['generator']['fonts']['max_size'] = 128
    cfg['generator']['image_size'] = [512, 512]
    cfg['generator']['num_workers'] = 2
    cfg['generator']['text']['min_length'] = 2
    cfg['generator']['text']['max_length'] = 20
    cfg['generator']['text']['min_words_per_line'] = 1
    cfg['generator']['text']['max_words_per_line'] = 3
    cfg['generator']['text']['max_lines'] = 1
    cfg['generator']['text']['case_distribution'] = {
        'lower': 0.50, 'upper': 0.20, 'mixed': 0.20, 'title': 0.10,
    }
    cfg['generator']['text']['rare_char_boost'] = 4.0
    cfg['generator']['text']['corpus_paths'] = [
        {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.30},
        {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
        {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
        {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.30},
    ]
    cfg['generator']['layout']['modes'] = {
        'horizontal': 0.50, 'rotated': 0.20, 'curved': 0.10,
        'perspective': 0.10, 'deformed': 0.10, 'paragraph': 0.0,
    }
    cfg['generator']['background']['natural_image_dirs'] = []
    test_cfg_path = '/kaggle/working/test_gen_config.yaml'
    Path(test_cfg_path).write_text(yaml.safe_dump(cfg))
    print(f'Generating {TEST_SET_SIZE} held-out samples at indices [{TEST_INDEX_OFFSET}, {TEST_INDEX_OFFSET + TEST_SET_SIZE})...')
    t0 = time.time()
    !python -m scripts.generate_dataset --config {test_cfg_path} --num-images {TEST_SET_SIZE} --output {TEST_DIR} --index-offset {TEST_INDEX_OFFSET}
    print(f'Generation done in {(time.time() - t0)/60:.1f} min')

n_imgs = len(os.listdir(f'{TEST_DIR}/images'))
print(f'Test set ready: {TEST_DIR} ({n_imgs} samples)')

## 5 / Build model + load weights

If multiple checkpoints were found, average them (equivalent to the offline `scripts.average_runs` flow). Otherwise load the single checkpoint.

In [ ]:
import yaml
from pathlib import Path
from segocr.models.unet import build_model

cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())
cfg['model']['architecture'] = 'unet'
cfg['model']['encoder'] = 'resnet18'
cfg['model']['encoder_weights'] = None   # weights come from checkpoint
cfg['model']['head_features'] = 64
cfg['model']['decoder_channels'] = [256, 128, 64, 32, 32]
cfg['model']['heads'] = {'semantic': True, 'affinity': True, 'direction': True}
cfg['model']['num_classes'] = 63
cfg['model']['input_size'] = [512, 512]

model = build_model(cfg['model']).to(DEVICE)

def _extract_state_dict(ckpt):
    if 'model' in ckpt:
        return ckpt['model']
    if 'ema' in ckpt:
        return ckpt['ema']
    return ckpt

if len(CHECKPOINT_PATHS) == 1:
    ckpt = torch.load(CHECKPOINT_PATHS[0], map_location=DEVICE)
    state = _extract_state_dict(ckpt)
    print(f'Loaded single checkpoint: {CHECKPOINT_PATHS[0]}')
else:
    states = [_extract_state_dict(torch.load(p, map_location='cpu')) for p in CHECKPOINT_PATHS]
    state = {}
    for key in states[0]:
        if not torch.is_tensor(states[0][key]) or not states[0][key].is_floating_point():
            state[key] = states[0][key]
            continue
        stacked = torch.stack([s[key].float() for s in states])
        state[key] = stacked.mean(dim=0).to(states[0][key].dtype)
    print(f'Averaged {len(states)} checkpoints into an ensemble.')

model.load_state_dict(state)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params/1e6:.2f}M params')

## 6 / Section A — Segmentation metrics

Streaming confusion matrix over the full test set → mIoU, fg_miou, binary_miou, per-class IoU. These are the headline segmentation numbers.

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
from segocr.training.dataset import SegOCRDataset, collate_fn
from segocr.training.evaluator import Evaluator
from segocr.evaluation.metrics import miou, fg_miou, binary_miou

NUM_CLASSES = 63
BATCH_SIZE = 8   # small for safety; eval is single-pass

test_ds = SegOCRDataset(TEST_DIR, split='train', train_aug=False)
# split='train' covers ALL files since the hash-split uses 5%% for val;
# we want the full 1K samples, not 950.
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

evaluator = Evaluator(num_classes=NUM_CLASSES, device=DEVICE)
evaluator.reset()
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        targets = batch['targets']['semantic'].to(DEVICE, non_blocking=True)
        outputs = model(images)
        preds = outputs['semantic'].argmax(dim=1)
        evaluator.update(preds, targets)

cm = evaluator.confusion_matrix.cpu().numpy()
seg_metrics = {
    'miou': miou(cm),
    'fg_miou': fg_miou(cm),
    'binary_miou': binary_miou(cm),
}
print('Segmentation metrics:')
for k, v in seg_metrics.items():
    print(f'  {k:15s} {v:.4f}')

# Per-class IoU
diag = np.diag(cm).astype(np.float64)
row = cm.sum(axis=1)
col = cm.sum(axis=0)
per_class_iou = (diag + 1e-9) / (row + col - diag + 1e-9)
print(f'\nPer-class IoU range: [{per_class_iou[1:].min():.3f}, {per_class_iou[1:].max():.3f}]')
print(f'Mean foreground IoU: {per_class_iou[1:].mean():.3f}')

## 7 / Section B — OCR-level metrics (decode → CER, exact-match)

For each test image: argmax + max-softmax → `cleanup_prediction` → `extract_instances` → `recover_text` → compare to ground-truth string. Reports CER, normalized edit distance, exact-match rate. The decoder uses our trained char-level segmentation, no language model.

In [ ]:
import json
import torch.nn.functional as F
from segocr.postprocessing.cleanup import cleanup_prediction
from segocr.postprocessing.instance_extraction import extract_instances
from segocr.postprocessing.reading_order import recover_text
from segocr.evaluation.metrics import cer, ned, exact_match

def _gt_string_from_metadata(meta_path):
    '''Reconstruct the ground-truth string from the per-char metadata
    using reading-order recovery (same pipeline as the prediction).'''
    with open(meta_path, encoding='utf-8') as f:
        m = json.load(f)
    # The on-disk metadata has each character's bbox + class_id.
    # Bbox is stored at supersampled scale — but the renderer scales
    # it back via the `scale` factor; here we just use the centroids
    # for ordering. As a robust fallback we just take char order as
    # rendered (Latin reading order is x-then-y).
    chars = m.get('characters', [])
    if not chars:
        return ''
    return ''.join(c['char'] for c in chars)

per_sample_results = []
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        outputs = model(images)
        logits = outputs['semantic']                          # (B, C, H, W)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1).cpu().numpy()             # (B, H, W)
        confs = probs.max(dim=1).values.cpu().numpy()         # (B, H, W)
        for i, info in enumerate(batch['metadata']):
            name = info['name']
            meta_path = f'{TEST_DIR}/metadata/{name}.json'
            gt = _gt_string_from_metadata(meta_path)
            clean = cleanup_prediction(preds[i], confs[i],
                                       threshold=0.5, min_component_area=20)
            instances = extract_instances(clean, min_size=8, max_size=256)
            pred_text = recover_text(instances)
            per_sample_results.append({
                'name': name,
                'gt': gt,
                'pred': pred_text,
                'cer': cer(pred_text, gt),
                'ned': ned(pred_text, gt),
                'exact_match': exact_match(pred_text, gt),
            })

import statistics
ocr_metrics = {
    'cer_mean':         statistics.mean(r['cer'] for r in per_sample_results),
    'ned_mean':         statistics.mean(r['ned'] for r in per_sample_results),
    'exact_match_rate': statistics.mean(r['exact_match'] for r in per_sample_results),
    'samples':          len(per_sample_results),
}
print('OCR metrics:')
for k, v in ocr_metrics.items():
    print(f'  {k:18s} {v:.4f}' if isinstance(v, float) else f'  {k:18s} {v}')

## 8 / Section C — Robustness slices

Stratifies the per-sample CER and exact-match rate by:
- **Font size bucket** (small / medium / large — derived from median character bbox height)
- **Layout mode** (horizontal / rotated / curved / perspective / deformed)

Reviewers love these — they show *where* the model fails, not just an aggregate number.

In [ ]:
from collections import defaultdict

def _load_meta(name):
    with open(f'{TEST_DIR}/metadata/{name}.json', encoding='utf-8') as f:
        return json.load(f)

def _font_size_bucket(meta):
    chars = meta.get('characters', [])
    if not chars:
        return 'unknown'
    heights = [c['bbox'][3] - c['bbox'][1] for c in chars]
    h = statistics.median(heights) if heights else 0
    if h < 40:    return 'small'
    if h < 80:    return 'medium'
    return 'large'

by_size = defaultdict(list)
by_layout = defaultdict(list)
for r in per_sample_results:
    meta = _load_meta(r['name'])
    by_size[_font_size_bucket(meta)].append(r)
    by_layout[meta.get('layout_mode', 'unknown')].append(r)

def _agg(group):
    return {
        'n':                len(group),
        'cer_mean':         statistics.mean(r['cer'] for r in group),
        'exact_match_rate': statistics.mean(r['exact_match'] for r in group),
    }

robustness = {
    'by_font_size': {k: _agg(v) for k, v in by_size.items()},
    'by_layout':    {k: _agg(v) for k, v in by_layout.items()},
}

print('--- By font size ---')
for k, m in sorted(robustness['by_font_size'].items()):
    print(f'  {k:8s}  n={m["n"]:4d}  CER={m["cer_mean"]:.3f}  EM={m["exact_match_rate"]:.3f}')
print('--- By layout mode ---')
for k, m in sorted(robustness['by_layout'].items()):
    print(f'  {k:12s}  n={m["n"]:4d}  CER={m["cer_mean"]:.3f}  EM={m["exact_match_rate"]:.3f}')

## 9 / Section D — Per-class IoU bar chart

One bar per character class (0=background, 1-26=A-Z, 27-52=a-z, 53-62=0-9). Drops the headline 'j' visualization that motivates the rare-character story in the paper.

In [ ]:
import matplotlib.pyplot as plt
import string

labels = ['bg'] + list(string.ascii_uppercase + string.ascii_lowercase + string.digits)
ious = per_class_iou
fig, ax = plt.subplots(figsize=(16, 4))
colors = ['gray'] + ['steelblue'] * 26 + ['darkorange'] * 26 + ['seagreen'] * 10
ax.bar(range(len(labels)), ious, color=colors)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('IoU')
ax.set_title('Per-class IoU')
ax.axhline(per_class_iou[1:].mean(), color='red', linestyle='--', linewidth=1, label='fg mean')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/kaggle/working/per_class_iou.png', dpi=150, bbox_inches='tight')
plt.show()

# Print bottom-10 classes — these are the rare ones the paper should call out
fg_iou_pairs = sorted(
    [(labels[i], float(per_class_iou[i])) for i in range(1, len(labels))],
    key=lambda kv: kv[1],
)
print('\nWorst-performing 10 classes:')
for ch, v in fg_iou_pairs[:10]:
    print(f'  {ch!r:6s}  IoU={v:.3f}')

## 10 / Section E — Qualitative gallery

8 success cases (exact_match=1) + 8 failure cases (highest CER). Use these in the paper's qualitative figure. Layout: 4×4 grid, each tile shows image + GT + predicted text.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

successes = [r for r in per_sample_results if r['exact_match'] == 1.0][:8]
failures = sorted(per_sample_results, key=lambda r: -r['cer'])[:8]
samples = successes + failures

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for ax, r in zip(axes.flat, samples, strict=False):
    img = Image.open(f'{TEST_DIR}/images/{r["name"]}.png')
    ax.imshow(img)
    status = 'OK' if r['exact_match'] == 1.0 else f'CER={r["cer"]:.2f}'
    ax.set_title(f'{status}\nGT:   {r["gt"][:30]}\nPred: {r["pred"][:30]}',
                 fontsize=8, loc='left')
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/qualitative_gallery.png', dpi=120, bbox_inches='tight')
plt.show()

## 11 / Section F — Inference speed

Warms the GPU then times forward passes for a batch of 8. Reports ms/image and FPS — basic efficiency numbers reviewers ask for.

In [ ]:
import time

# Warmup
warmup_batch = next(iter(test_loader))
warmup_images = warmup_batch['image'].to(DEVICE)
with torch.no_grad():
    for _ in range(3):
        _ = model(warmup_images)
torch.cuda.synchronize()

n_batches = 0
n_images = 0
t0 = time.perf_counter()
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(DEVICE, non_blocking=True)
        _ = model(images)
        n_batches += 1
        n_images += images.shape[0]
        if n_batches >= 25:    # ~200 images is enough for a stable mean
            break
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
ms_per_image = (elapsed / n_images) * 1000
fps = n_images / elapsed
speed = {'ms_per_image': ms_per_image, 'fps': fps, 'n_images': n_images}
print(f'Speed: {ms_per_image:.1f} ms/image  ({fps:.1f} FPS, batch={BATCH_SIZE})')

## 12 / Section G — Export results

Dumps everything to `/kaggle/working/results.json` (machine-readable for the paper-build pipeline) and writes a markdown table you can paste directly into the LaTeX/Markdown manuscript.

In [ ]:
import json

results = {
    'checkpoints_used': CHECKPOINT_PATHS,
    'n_test_samples': ocr_metrics['samples'],
    'segmentation': seg_metrics,
    'ocr': ocr_metrics,
    'robustness': robustness,
    'speed': speed,
    'per_class_iou': {labels[i]: float(per_class_iou[i]) for i in range(len(labels))},
    'model_params_M': float(n_params / 1e6),
}

with open('/kaggle/working/results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print('Wrote /kaggle/working/results.json')

# Build a paper-ready markdown table
md_lines = [
    '## SegOCR results',
    '',
    '### Segmentation',
    '',
    '| Metric | Value |',
    '|---|---|',
    f'| mIoU | {seg_metrics["miou"]:.3f} |',
    f'| fg mIoU | {seg_metrics["fg_miou"]:.3f} |',
    f'| Binary mIoU | {seg_metrics["binary_miou"]:.3f} |',
    '',
    '### OCR',
    '',
    '| Metric | Value |',
    '|---|---|',
    f'| CER (mean) | {ocr_metrics["cer_mean"]:.3f} |',
    f'| NED (mean) | {ocr_metrics["ned_mean"]:.3f} |',
    f'| Exact match | {ocr_metrics["exact_match_rate"]:.3f} |',
    '',
    '### Robustness — by font size',
    '',
    '| Bucket | N | CER | Exact match |',
    '|---|---|---|---|',
]
for k in ('small', 'medium', 'large'):
    m = robustness['by_font_size'].get(k)
    if m:
        md_lines.append(f'| {k} | {m["n"]} | {m["cer_mean"]:.3f} | {m["exact_match_rate"]:.3f} |')
md_lines += ['', '### Robustness — by layout mode', '', '| Mode | N | CER | Exact match |', '|---|---|---|---|']
for k, m in sorted(robustness['by_layout'].items()):
    md_lines.append(f'| {k} | {m["n"]} | {m["cer_mean"]:.3f} | {m["exact_match_rate"]:.3f} |')
md_lines += ['', '### Efficiency', '', '| Metric | Value |', '|---|---|',
             f'| Params (M) | {results["model_params_M"]:.2f} |',
             f'| Inference ms/image | {speed["ms_per_image"]:.1f} |',
             f'| FPS @ batch={BATCH_SIZE} | {speed["fps"]:.1f} |']

with open('/kaggle/working/results_paper_table.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(md_lines))
print('Wrote /kaggle/working/results_paper_table.md')
print()
print('\n'.join(md_lines))